In [27]:
# LOAD SEMUA LIBRARY YANG DIPERLUKAN

import pandas as pd
import numpy as np
from scipy.spatial import KDTree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, f1_score, confusion_matrix, mean_absolute_error, r2_score)
from sklearn.utils.class_weight import compute_class_weight
import joblib
import warnings
import os
warnings.filterwarnings('ignore')

In [28]:
# LOAD SEMUA DATASET YANG DIPERLUKAN

print("=" * 60)
print("| UMKM LOCATION PREDICTION ")
print("=" * 60)

RAW_DATASETS_PATH = "./raw_datasets"

# Load dataset POI (point of interest)
df_halte = pd.read_csv(RAW_DATASETS_PATH + "/poi/data_titik_halte_bus_bandung.csv")
df_kafe = pd.read_csv(RAW_DATASETS_PATH + "/poi/data_titik_kafe_bandung.csv")
df_kampus = pd.read_csv(RAW_DATASETS_PATH + "/poi/data_titik_kampus_bandung.csv")
df_pasar = pd.read_csv(RAW_DATASETS_PATH + "/poi/data_titik_pasar_bandung.csv")
df_restoran = pd.read_csv(RAW_DATASETS_PATH + "/poi/data_titik_restoran_bandung.csv")
df_retail_bandung = pd.read_csv(RAW_DATASETS_PATH + "/poi/data_retail_bandung.csv")
df_sekolah = pd.read_csv(RAW_DATASETS_PATH + "/poi/data_titik_sekolah_bandung.csv")

# Load dataset kepadatan penduduk
df_kepadatan_penduduk_raw = pd.read_csv(RAW_DATASETS_PATH + "/demografi/data_kepadatan_penduduk_bandung.csv", skiprows=2, header=None, names=["kecamatan", "kepadatan"])

# merapihkan dataframe kepadatan, menghapus baris kosong dan non-numerik
df_kepadatan_penduduk = df_kepadatan_penduduk_raw[
    df_kepadatan_penduduk_raw["kecamatan"].notna() &
    df_kepadatan_penduduk_raw["kepadatan"].notna() &
    (df_kepadatan_penduduk_raw["kepadatan"] != "-") &
    (df_kepadatan_penduduk_raw["kecamatan"].str.strip() != "")    
].copy()
df_kepadatan_penduduk["kepadatan"] = pd.to_numeric(df_kepadatan_penduduk["kepadatan"], errors="coerce")
df_kepadatan_penduduk = df_kepadatan_penduduk.dropna()
df_kepadatan_penduduk["kecamatan"] = df_kepadatan_penduduk["kecamatan"].str.strip().str.lower()

print(f"  Halte Bus      : {len(df_halte)} titik")
print(f"  Kafe           : {len(df_kafe)} titik")
print(f"  Kampus         : {len(df_kampus)} titik")
print(f"  Pasar          : {len(df_pasar)} titik")
print(f"  Restoran       : {len(df_restoran)} titik")
print(f"  Sekolah        : {len(df_sekolah)} titik")
print(f"  Retail         : {len(df_retail_bandung)} titik")
print(f"  Kepadatan      : {len(df_kepadatan_penduduk)} titik")

| UMKM LOCATION PREDICTION 
  Halte Bus      : 39 titik
  Kafe           : 412 titik
  Kampus         : 106 titik
  Pasar          : 49 titik
  Restoran       : 944 titik
  Sekolah        : 567 titik
  Retail         : 257 titik
  Kepadatan      : 30 titik


In [29]:
# BUILD KD TREES (untuk mempercepat pencarian terdekat)

def build_kdtree(df):
    coords = df[["lat", "lon"]].values
    return KDTree(coords)

tree_halte = build_kdtree(df_halte)
tree_kafe = build_kdtree(df_kafe)
tree_kampus = build_kdtree(df_kampus)
tree_pasar = build_kdtree(df_pasar)
tree_restoran = build_kdtree(df_restoran)
tree_sekolah = build_kdtree(df_sekolah)

print("Build KD Trees successful.")

def meter_to_deg(meter):
    return meter / 111_000

Build KD Trees successful.


In [39]:
# GENERATE GRID BANDUNG

# Bounding box Kota Bandung
LAT_MIN, LAT_MAX = -6.980, -6.855
LON_MIN, LON_MAX = 107.550, 107.725
GRID_STEP = 0.008

lats = np.arange(LAT_MIN, LAT_MAX, GRID_STEP)
lons = np.arange(LON_MIN, LON_MAX, GRID_STEP)

grid_points = [(lat, lon) for lat in lats for lon in lons]
print(f" Total grid points: {len(grid_points)}")

 Total grid points: 352


In [31]:
# FEATURE ENGINEERING

# Mapping koordinat ke kecamatan (simplified polygon-free approach)
# Menggunakan nearest neighbor dari centroid kecamatan yang diketahui
KECAMATAN_CENTROIDS = {
    "bojongloa kaler":   (-6.934, 107.585),
    "babakan ciparay":   (-6.947, 107.594),
    "bandung kulon":     (-6.942, 107.573),
    "astanaanyar":       (-6.935, 107.600),
    "regol":             (-6.939, 107.614),
    "lengkong":          (-6.928, 107.621),
    "batununggal":       (-6.935, 107.633),
    "kiaracondong":      (-6.924, 107.644),
    "antapani":          (-6.918, 107.662),
    "mandalajati":       (-6.903, 107.669),
    "arcamanik":         (-6.907, 107.685),
    "ujungberung":       (-6.906, 107.706),
    "cibiru":            (-6.904, 107.723),
    "panyileukan":       (-6.920, 107.712),
    "cinambo":           (-6.921, 107.698),
    "gedebage":          (-6.951, 107.693),
    "rancasari":         (-6.956, 107.660),
    "buahbatu":          (-6.958, 107.641),
    "bandung kidul":     (-6.952, 107.621),
    "bojongloa kidul":   (-6.950, 107.600),
    "andir":             (-6.912, 107.587),
    "cicendo":           (-6.907, 107.597),
    "sumur bandung":     (-6.918, 107.609),
    "bandung wetan":     (-6.905, 107.618),
    "cibeunying kidul":  (-6.912, 107.636),
    "cibeunying kaler":  (-6.897, 107.637),
    "coblong":           (-6.893, 107.617),
    "sukajadi":          (-6.893, 107.597),
    "sukasari":          (-6.882, 107.583),
    "cidadap":           (-6.868, 107.594),
}

kec_names  = list(KECAMATAN_CENTROIDS.keys())

kec_coords = np.array(list(KECAMATAN_CENTROIDS.values()))
kec_tree   = KDTree(kec_coords)

def get_kecamatan(lat, lon):
    _, idx = kec_tree.query([lat, lon])
    return kec_names[idx]

def extract_features(lat, lon):
    point = np.array([lat, lon])

    # Radius queries menggunakan KD-Tree
    r500  = meter_to_deg(500)
    r1000 = meter_to_deg(1000)

    n_sekolah    = len(tree_sekolah.query_ball_point(point, r500))
    n_pasar      = len(tree_pasar.query_ball_point(point, r1000))
    n_kampus     = len(tree_kampus.query_ball_point(point, r1000))
    n_restoran   = len(tree_restoran.query_ball_point(point, r500))
    n_kafe       = len(tree_kafe.query_ball_point(point, r500))
    n_halte      = len(tree_halte.query_ball_point(point, r500))

    # Kompetitor = kafe + restoran (proxy untuk persaingan F&B)
    n_kompetitor = n_kafe + n_restoran

    # Kepadatan penduduk
    kec = get_kecamatan(lat, lon)
    row = df_kepadatan_penduduk[df_kepadatan_penduduk["kecamatan"] == kec]
    kepadatan = float(row["kepadatan"].values[0]) if len(row) > 0 else df_kepadatan_penduduk["kepadatan"].mean()

    return {
        "lat": lat,
        "lon": lon,
        "kecamatan": kec,
        "kepadatan_penduduk": kepadatan,
        "jumlah_sekolah_500m": n_sekolah,
        "jumlah_pasar_1km": n_pasar,
        "jumlah_kampus_1km": n_kampus,
        "jumlah_restoran_500m": n_restoran,
        "jumlah_kafe_500m": n_kafe,
        "jumlah_halte_500m": n_halte,
        "jumlah_kompetitor_500m": n_kompetitor,
    }

# ekstrak fitur per titik grid
records = []
for i, (lat, lon) in enumerate(grid_points):
    records.append(extract_features(lat, lon))
    if (i + 1) % 100 == 0:
        print(f"  Progress: {i+1}/{len(grid_points)}")

df = pd.DataFrame(records)
print(f"  Feature extraction selesai: {len(df)} titik")
print(f"\n  Sample data:")
print(df[["lat","lon","kecamatan","kepadatan_penduduk",
          "jumlah_sekolah_500m","jumlah_kampus_1km",
          "jumlah_restoran_500m","jumlah_kompetitor_500m"]].head(10).to_string())

  Progress: 100/352
  Progress: 200/352
  Progress: 300/352
  Feature extraction selesai: 352 titik

  Sample data:
    lat      lon        kecamatan  kepadatan_penduduk  jumlah_sekolah_500m  jumlah_kampus_1km  jumlah_restoran_500m  jumlah_kompetitor_500m
0 -6.98  107.550    bandung kulon             21397.0                    0                  0                     0                       0
1 -6.98  107.558    bandung kulon             21397.0                    0                  0                     0                       0
2 -6.98  107.566    bandung kulon             21397.0                    0                  0                     0                       0
3 -6.98  107.574    bandung kulon             21397.0                    0                  0                     0                       0
4 -6.98  107.582  bojongloa kidul             14278.0                    0                  0                     0                       0
5 -6.98  107.590  bojongloa kidul           

In [32]:
# GENERATE LABEL SINTETIS

"""
Note: karena masih sintetis, skor akan dipertimbangkan dari:
- Kepadatan penduduk tinggi -> baik
- Banyak kampus -> baik (arus mahasiswa)
- Banyak pasar -> baik (area komersial)
- Banyak restoran -> baik (area aktif)
- Terlalu banyak kompetitor -> kurang baik
- Ada halte bus -> baik (aksesibilitas) 
"""

def compute_success_score(row):
    # Normalisasi per fitur (0-1)
    norm_kepadatan   = min(row["kepadatan_penduduk"] / 45000, 1.0)
    norm_sekolah     = min(row["jumlah_sekolah_500m"] / 10, 1.0)
    norm_pasar       = min(row["jumlah_pasar_1km"] / 5, 1.0)
    norm_kampus      = min(row["jumlah_kampus_1km"] / 5, 1.0)
    norm_restoran    = min(row["jumlah_restoran_500m"] / 30, 1.0)
    norm_halte       = min(row["jumlah_halte_500m"] / 5, 1.0)
    # Kompetitor bersifat negatif
    norm_kompetitor  = min(row["jumlah_kompetitor_500m"] / 50, 1.0)

    # Weighted score
    score = (
        norm_kepadatan  * 0.25 +
        norm_kampus     * 0.20 +
        norm_restoran   * 0.15 +
        norm_pasar      * 0.15 +
        norm_sekolah    * 0.10 +
        norm_halte      * 0.10 +
        (1 - norm_kompetitor) * 0.05   # inverse: kompetitor tinggi = kurang baik
    )

    # Skala ke 0-100 + noise realistis
    noise = np.random.normal(0, 3)
    return float(np.clip(score * 100 + noise, 0, 100))

np.random.seed(42)
df["success_score"] = df.apply(compute_success_score, axis=1)

# Buat kategori (untuk klasifikasi)
def score_to_category(score):
    if score >= 71:
        return "Tinggi"
    elif score >= 41:
        return "Sedang"
    else:
        return "Rendah"

df["kategori"] = df["success_score"].apply(score_to_category)

print(f"\n  Distribusi kategori:")
print(df["kategori"].value_counts().to_string())
print(f"\n  Statistik success_score:")
print(df["success_score"].describe().round(2).to_string())


  Distribusi kategori:
kategori
Rendah    310
Sedang     42

  Statistik success_score:
count    352.00
mean      21.10
std       13.34
min        3.27
25%       12.01
50%       16.36
75%       26.26
max       66.32


In [33]:
# Data splitting dan Persiapan Fitur

FEATURE_COLS = [
    "kepadatan_penduduk",
    "jumlah_sekolah_500m",
    "jumlah_pasar_1km",
    "jumlah_kampus_1km",
    "jumlah_restoran_500m",
    "jumlah_halte_500m",
    "jumlah_kompetitor_500m",
]

# X (Features)
X = df[FEATURE_COLS].values

# Y (Label/Target)
y_reg = df["success_score"].values        # untuk Regression
y_clf = df["kategori"].values             # untuk Classification

# Encode label untuk klasifikasi
le = LabelEncoder()
y_clf_enc = le.fit_transform(y_clf)       # Rendah=0, Sedang=1, Tinggi=2
print(f"  Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Split: 70% train, 15% val, 15% test (STRATIFIED)
X_temp, X_test, y_reg_temp, y_reg_test, y_clf_temp, y_clf_test = train_test_split(
    X, y_reg, y_clf_enc,
    test_size=0.15, random_state=42, stratify=y_clf_enc
)

X_train, X_val, y_reg_train, y_reg_val, y_clf_train, y_clf_val = train_test_split(
    X_temp, y_reg_temp, y_clf_temp,
    test_size=0.1765,  # 0.15/0.85 sekitar 0.1765 --> hasil akhir 15% dari total
    random_state=42, stratify=y_clf_temp
)

print(f"  Train set : {len(X_train)} sampel ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Val set   : {len(X_val)} sampel ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test set  : {len(X_test)} sampel ({len(X_test)/len(X)*100:.1f}%)")

# Cek distribusi kelas di train set
unique, counts = np.unique(y_clf_train, return_counts=True)
print(f"\n  Distribusi kelas di train set:")
for cls, cnt in zip(le.classes_[unique], counts):
    print(f"    {cls}: {cnt} ({cnt/len(y_clf_train)*100:.1f}%)")

  Label encoding: {'Rendah': np.int64(0), 'Sedang': np.int64(1)}
  Train set : 246 sampel (69.9%)
  Val set   : 53 sampel (15.1%)
  Test set  : 53 sampel (15.1%)

  Distribusi kelas di train set:
    Rendah: 216 (87.8%)
    Sedang: 30 (12.2%)


In [34]:
# PENANGANAN IMBALANCE

# Cek rasio imbalance
counts_dict = dict(zip(unique, counts))
max_count = max(counts_dict.values())
min_count = min(counts_dict.values())
ratio = max_count / min_count

print(f"  Imbalance ratio: {ratio:.2f}:1")

if ratio > 3:
    print(" Ratio > 3:1, menggunakan SMOTE")
    try:
        from imblearn.over_sampling import SMOTE
        smote = SMOTE(random_state=42, k_neighbors=min(5, min_count-1))
        X_train_bal, y_clf_train_bal = smote.fit_resample(X_train, y_clf_train)
        # Untuk regression, gunakan nilai rata-rata per kelas
        y_reg_train_bal = np.array([
            y_reg_train[y_clf_train == c].mean() if c in y_clf_train else 50.0
            for c in y_clf_train_bal
        ])
        print(f" SMOTE: {len(X_train)} -> {len(X_train_bal)} sampel")
    except ImportError:
        print(" imbalanced-learn tidak tersedia, fallback ke class_weight")
        X_train_bal, y_clf_train_bal, y_reg_train_bal = X_train, y_clf_train, y_reg_train
else:
    print(f" Ratio {ratio:.2f}:1 (≤ 3:1), menggunakan class_weight='balanced'")
    X_train_bal, y_clf_train_bal, y_reg_train_bal = X_train, y_clf_train, y_reg_train

  Imbalance ratio: 7.20:1
 Ratio > 3:1, menggunakan SMOTE
 SMOTE: 246 -> 432 sampel


In [40]:
#  PROSES TRAINING

# Training model Random Forest Classifier
print("\n  [1] Training Random Forest Classifier...")
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train_bal, y_clf_train_bal)

y_val_pred_clf = clf.predict(X_val)
f1_val = f1_score(y_clf_val, y_val_pred_clf, average="weighted")
print(f" Classifier F1-Score (Validation): {f1_val:.4f}")

print("\n  [2] Training Random Forest Regressor...")
reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)   
reg.fit(X_train_bal, y_reg_train_bal)

y_val_pred_reg = reg.predict(X_val)
mae_val = mean_absolute_error(y_reg_val, y_val_pred_reg)
r2_val  = r2_score(y_reg_val, y_val_pred_reg)
print(f" Regressor MAE (Validation)  : {mae_val:.2f}")
print(f" Regressor R² (Validation)   : {r2_val:.4f}")


  [1] Training Random Forest Classifier...
 Classifier F1-Score (Validation): 0.9818

  [2] Training Random Forest Regressor...
 Regressor MAE (Validation)  : 6.13
 Regressor R² (Validation)   : 0.6522


In [36]:
# FINAL EVALUATION ON TEST SET

# Classifier
y_test_pred_clf = clf.predict(X_test)
f1_test = f1_score(y_clf_test, y_test_pred_clf, average="weighted")

print(f"\n  === CLASSIFICATION REPORT (Test Set) ===")
print(classification_report(
    y_clf_test, y_test_pred_clf,
    target_names=le.classes_
))
print(f"  F1-Score Weighted (Test): {f1_test:.4f}")

# Regressor
y_test_pred_reg = reg.predict(X_test)
mae_test = mean_absolute_error(y_reg_test, y_test_pred_reg)
r2_test  = r2_score(y_reg_test, y_test_pred_reg)
print(f"\n  === REGRESSION METRICS (Test Set) ===")
print(f"  MAE : {mae_test:.2f} poin")
print(f"  R²  : {r2_test:.4f}")


  === CLASSIFICATION REPORT (Test Set) ===
              precision    recall  f1-score   support

      Rendah       0.96      1.00      0.98        47
      Sedang       1.00      0.67      0.80         6

    accuracy                           0.96        53
   macro avg       0.98      0.83      0.89        53
weighted avg       0.96      0.96      0.96        53

  F1-Score Weighted (Test): 0.9589

  === REGRESSION METRICS (Test Set) ===
  MAE : 5.61 poin
  R²  : 0.6558


In [ ]:
# SHAP 
# 3 VARIABLE TERATAS & TERPENTING (IMPORTANCE)


# Feature Importance (global - statis)
importances = clf.feature_importances_
fi_df = pd.DataFrame({
    "fitur": FEATURE_COLS,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\n  === GLOBAL FEATURE IMPORTANCE ===")
for _, row in fi_df.iterrows():
    bar = "█" * int(row["importance"] * 50)
    print(f"  {row['fitur']:35s} {bar} {row['importance']:.4f}")

# SHAP Demo (lokal - dinamis per sampel)
print("\n  === SHAP VALUES DEMO (3 lokasi berbeda) ===")
try:
    import shap
    explainer = shap.TreeExplainer(clf)
    # Ambil 3 sampel dari test set sebagai demo
    sample_indices = [0, len(X_test)//2, -1]
    X_demo = X_test[sample_indices]

    shap_values = explainer.shap_values(X_demo)

    for i, idx in enumerate(sample_indices):
        pred_class = le.classes_[y_test_pred_clf[idx]]
        pred_score = y_test_pred_reg[idx]
        print(f"\n  Lokasi {i+1} | Prediksi: {pred_class} | Skor: {pred_score:.1f}/100")
        print(f"  Koordinat: {X_test[idx][:2] if len(X_test[idx]) > 1 else 'N/A'}")

        # SHAP values untuk kelas yang diprediksi
        pred_class_idx = list(le.classes_).index(pred_class)
        if isinstance(shap_values, list):
            sv = shap_values[pred_class_idx][i]
        else:
            sv = shap_values[i]

        # Pastikan shap values adalah 1D array
        sv = np.array(sv).flatten()
        shap_per_feat = [(feat, float(val)) for feat, val in zip(FEATURE_COLS, sv)]
        shap_per_feat.sort(key=lambda x: abs(x[1]), reverse=True)

        print(f"  Top 3 Faktor (SHAP):")
        for j, (feat, val) in enumerate(shap_per_feat[:3]):
            arah = "▲ Positif" if val > 0 else "▼ Negatif"
            print(f"    {j+1}. {feat:35s} {arah} ({val:+.4f})")

except ImportError:
    print(" SHAP tidak tersedia, skip demo")
    explainer = None


  === GLOBAL FEATURE IMPORTANCE ===
  jumlah_pasar_1km                    ████████████ 0.2484
  jumlah_kampus_1km                   ███████████ 0.2217
  jumlah_sekolah_500m                 ██████████ 0.2132
  jumlah_kompetitor_500m              ███████ 0.1435
  jumlah_restoran_500m                █████ 0.1057
  kepadatan_penduduk                  ███ 0.0646
  jumlah_halte_500m                    0.0028

  === SHAP VALUES DEMO (3 lokasi berbeda) ===

  Lokasi 1 | Prediksi: Rendah | Skor: 17.2/100
  Koordinat: [1.1398e+04 3.0000e+00]
  Top 3 Faktor (SHAP):
    1. jumlah_kompetitor_500m              ▲ Positif (+0.2190)
    2. jumlah_halte_500m                   ▼ Negatif (-0.0750)
    3. jumlah_restoran_500m                ▲ Positif (+0.0750)

  Lokasi 2 | Prediksi: Rendah | Skor: 17.2/100
  Koordinat: [13472.     0.]
  Top 3 Faktor (SHAP):
    1. jumlah_halte_500m                   ▼ Negatif (-0.1262)
    2. jumlah_restoran_500m                ▲ Positif (+0.1262)
    3. jumlah_kompetito

In [41]:
# SAVE MODEL & DATA

OUTPUT_PATH = "./outputs/"
os.makedirs(OUTPUT_PATH + "train_data", exist_ok=True)
os.makedirs(OUTPUT_PATH + "test_data", exist_ok=True)
os.makedirs(OUTPUT_PATH + "model", exist_ok=True)

# save model
joblib.dump(clf, OUTPUT_PATH + "model/rf_classifier.pkl")
joblib.dump(reg, OUTPUT_PATH + "model/rf_regressor.pkl")
joblib.dump(le,  OUTPUT_PATH + "model/label_encoder.pkl")
if explainer:
    joblib.dump(explainer, OUTPUT_PATH + "model/shap_explainer.pkl")

# save train_data dan test_data terpisah (constraint bab 3)
train_indices = np.concatenate([
    np.where(np.isin(df.index, range(len(X_train_bal))))[0]
])

# save as csv dengan semua fitur
df_full = df.copy()
df_full["split"] = "train"

# Recreate split indices dari df
_, test_idx = train_test_split(
    df.index, test_size=0.15, random_state=42,
    stratify=df["kategori"]
)
train_idx = df.index.difference(test_idx)

df.loc[train_idx].to_csv(OUTPUT_PATH + "train_data/train_data.csv", index=False)
df.loc[test_idx].to_csv(OUTPUT_PATH + "test_data/test_data.csv", index=False)

# output path
print(f" Output Path Location")
print(f" rf_classifier.pkl: {OUTPUT_PATH}model/")
print(f" rf_regressor.pkl : {OUTPUT_PATH}model/")
print(f" label_encoder.pkl: {OUTPUT_PATH}model/")
print(f" train_data.csv   : {OUTPUT_PATH}train_data/")
print(f" test_data.csv    : {OUTPUT_PATH}test_data/")

# validasi no data leakage (constraint bab 3)
train_df_saved = pd.read_csv(OUTPUT_PATH + "train_data/train_data.csv")
test_df_saved  = pd.read_csv(OUTPUT_PATH + "test_data/test_data.csv")

train_coords = set(zip(train_df_saved["lat"].round(6), train_df_saved["lon"].round(6)))
test_coords  = set(zip(test_df_saved["lat"].round(6), test_df_saved["lon"].round(6)))
overlap      = train_coords.intersection(test_coords)

print(f"\n  === DATA LEAKAGE VALIDATION ===")
print(f"  Train set : {len(train_df_saved)} baris")
print(f"  Test set  : {len(test_df_saved)} baris")
print(f"  Overlap   : {len(overlap)} baris")
if len(overlap) == 0:
    print(f"  No Data Leakage")
else:
    print(f" Warning: {len(overlap)} overlapping records found")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Total data points      : {len(df)}")
print(f"  Features used          : {len(FEATURE_COLS)}")
print(f"  Train / Val / Test     : {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"  Imbalance handling     : {'SMOTE' if ratio > 3 else 'class_weight=balanced'}")
print(f"  Classifier F1 (test)   : {f1_test:.4f}")
print(f"  Regressor MAE (test)   : {mae_test:.2f} poin")
print(f"  Regressor R² (test)    : {r2_test:.4f}")
print(f"  Data leakage           : {'None' if len(overlap) == 0 else 'Detected'}")
print("=" * 60)


 Output Path Location
 rf_classifier.pkl: ./outputs/model/
 rf_regressor.pkl : ./outputs/model/
 label_encoder.pkl: ./outputs/model/
 train_data.csv   : ./outputs/train_data/
 test_data.csv    : ./outputs/test_data/

  === DATA LEAKAGE VALIDATION ===
  Train set : 299 baris
  Test set  : 53 baris
  Overlap   : 0 baris
  No Data Leakage

  TRAINING SUMMARY
  Total data points      : 352
  Features used          : 7
  Train / Val / Test     : 246 / 53 / 53
  Imbalance handling     : SMOTE
  Classifier F1 (test)   : 0.9589
  Regressor MAE (test)   : 5.61 poin
  Regressor R² (test)    : 0.6558
  Data leakage           : None
